# exp04 - Perakitan tabel naskah dari berkas hasil

Notebook ini **tidak menjalankan model apa pun**. Ia hanya membaca `results/*.csv`
yang dihasilkan exp01-exp03 dan merakitnya menjadi tabel siap-tempel (Markdown dan
LaTeX). Pemisahan ini disengaja: angka pada naskah tidak boleh pernah diketik ulang
dengan tangan - itulah asal ketidakkonsistenan RMSE/MSE yang ditemukan reviewer.

Jalankan exp01, exp02, dan exp03 terlebih dahulu.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
from pathlib import Path

from src.experiments import protocol as P

RESULTS = Path("../results")
OUT = RESULTS / "paper_tables"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 50)

def load(name):
    path = RESULTS / f"{name}.csv"
    if not path.exists():
        print(f"[lewati] {path} belum ada - jalankan notebook eksperimennya dulu")
        return None
    return pd.read_csv(path)

daily   = load("exp01_pharma_daily")
weekly  = load("exp02_pharma_weekly")
rossman = load("exp03_rossmann_leakage_ablation")

In [2]:
def emit(df, stem, caption, float_fmt="%.4f"):
    """Tulis satu tabel sebagai CSV + Markdown + LaTeX dengan nama yang sama."""
    df.to_csv(OUT / f"{stem}.csv")
    (OUT / f"{stem}.md").write_text(df.to_markdown(floatfmt=".4f"), encoding="utf-8")
    (OUT / f"{stem}.tex").write_text(
        df.to_latex(float_format=float_fmt, caption=caption, label=f"tab:{stem}",
                    escape=True), encoding="utf-8")
    print(f"ditulis: {stem}.csv / .md / .tex   -- {caption}")
    return df

## Tabel 1 - PharmaSales harian: semua model di bawah protokol tunggal

In [3]:
MODEL_ORDER = ["Naive", "SeasonalNaive(s=7)", "SeasonalNaive(s=52)", "ARIMA(5,1,0)",
               "LR", "GRNN", "P_NN", "RBFNN", "XGBoost", "LR+XGB (average)",
               "LR-XGB (residual)"]

def main_table(df, metric="test_RMSE"):
    pivot = df.pivot_table(index=["category", "feature_set"], columns="model",
                           values=metric)
    return pivot[[m for m in MODEL_ORDER if m in pivot.columns]]

if daily is not None:
    t1 = emit(main_table(daily).round(4), "tab1_pharma_daily_rmse",
              "RMSE test PharmaSales harian di bawah protokol tunggal "
              "(split 70/15/15, tuning hanya pada validation, seed 42).")
    display(t1)

ditulis: tab1_pharma_daily_rmse.csv / .md / .tex   -- RMSE test PharmaSales harian di bawah protokol tunggal (split 70/15/15, tuning hanya pada validation, seed 42).


model                   Naive  SeasonalNaive(s=7)  ARIMA(5,1,0)       LR     GRNN     P_NN    RBFNN  XGBoost  LR+XGB (average)  LR-XGB (residual)
category feature_set                                                                                                                             
M01AB    A_lag1        4.1726              4.0449        2.9932   2.9268   2.9140   3.1827   2.9229   3.0326            2.9724             3.0338
         B_rich        4.1726              4.0449           NaN   2.9268   2.9140   3.1827   2.9229   3.0326            2.9724             3.0338
M01AE    A_lag1        2.8509              2.8647        2.4341   2.3220   2.3544   2.5472   2.3035   2.3064            2.3109             2.3027
         B_rich        2.8509              2.8647           NaN   2.2025   2.2942   2.3599   2.2625   2.2365            2.2110             2.1891
N02BA    A_lag1        2.7857              2.6936        2.0357   2.1554   2.1503   2.2020   2.1551   2.1633            2.1545             2.1631
         B_rich        2.7857              2.6936           NaN   2.1554   2.1503   2.2020   2.1551   2.1633            2.1545             2.1631
N02BE    A_lag1       16.3589             16.3242       17.3057  14.3395  14.3567  14.8283  14.2656  14.2650           14.2309            14.2362
         B_rich       16.3589             16.3242           NaN  14.3395  14.3567  14.8283  14.2656  14.2650           14.2309            14.2362
N05B     A_lag1        5.8983              5.7878        4.3257   4.4338   4.3883   5.9764   4.4381   4.4584            4.4741             4.4602
         B_rich        5.8983              5.7878           NaN   4.4338   4.3883   5.9764   4.4381   4.4584            4.4741             4.4602
N05C     A_lag1        1.6621              1.6275        1.1353   1.1452   1.1433   1.2709   1.1442   1.1448            1.1446             1.1449
         B_rich        1.6621              1.6275           NaN   1.1471   1.1455   1.2709   1.1461   1.1599            1.1558             1.1946
R03      A_lag1       10.8952             10.8831        8.5551   8.2880   8.3497  10.2576   8.3298   8.3909            8.3223             8.4016
         B_rich       10.8952             10.8831           NaN   8.2880   8.3497  10.2576   8.3298   8.3909            8.3223             8.4016
R06      A_lag1        3.0094              3.1969        3.2667   2.5460   2.5546   3.1865   2.5512   2.5482            2.5434             2.5461
         B_rich        3.0094              3.1969           NaN   2.5460   2.5546   3.1865   2.5512   2.5482            2.5434             2.5461

## Tabel 2 - PharmaSales mingguan

In [4]:
if weekly is not None:
    t2 = emit(main_table(weekly).round(4), "tab2_pharma_weekly_rmse",
              "RMSE test PharmaSales mingguan di bawah protokol tunggal.")
    display(t2)

ditulis: tab2_pharma_weekly_rmse.csv / .md / .tex   -- RMSE test PharmaSales mingguan di bawah protokol tunggal.


model                   Naive  SeasonalNaive(s=52)  ARIMA(5,1,0)       LR     GRNN     P_NN    RBFNN  XGBoost  LR+XGB (average)  LR-XGB (residual)
category feature_set                                                                                                                              
M01AB    A_lag1       10.9205              10.7591        8.8008   8.7193   8.7884   9.2375   8.7108   8.8864            8.6943             8.8407
         B_rich       10.9205              10.7591           NaN   8.6713   8.7781   9.1831   8.6004   9.0576            8.8860             9.1217
M01AE    A_lag1        9.9500              10.5276       10.2251   9.2275  10.2813  10.6194  10.2508   9.6425            9.3917             9.4054
         B_rich        9.9500              10.5276           NaN   9.2275  10.2813  10.6194  10.2508   9.6425            9.3917             9.4054
N02BA    A_lag1        6.2395               7.2071        6.5336   6.8662   7.1841   6.2623   6.7827   7.1617            6.9652             7.1327
         B_rich        6.2395               7.2071           NaN   6.8662   7.1841   6.2623   6.7827   7.1617            6.9652             7.1327
N02BE    A_lag1       53.5960              61.8659       85.9269  51.3467  50.3296  55.3193  51.1346  48.7267           49.3749            49.5183
         B_rich       53.5960              61.8659           NaN  51.3467  50.3296  55.3193  51.1346  48.7267           49.3749            49.5183
N05B     A_lag1       15.4933              21.3796       12.3257  12.9702  12.7175  12.9179  12.5360  13.6580           13.1263            13.2927
         B_rich       15.4933              21.3796           NaN  12.9702  12.7175  12.9179  12.5360  13.6580           13.1263            13.2927
N05C     A_lag1        4.0332               4.8831        2.8170   2.9551   2.9744   3.4476   3.0717   3.1122            3.0160             3.1126
         B_rich        4.0332               4.8831           NaN   2.9410   2.8868   3.3635   2.9331   2.9094            3.0543             2.9432
R03      A_lag1       29.1849              33.6463       43.1389  26.7117  26.9553  27.1508  27.8197  26.7305           26.3901            26.4546
         B_rich       29.1849              33.6463           NaN  26.7117  26.9553  27.1508  27.8197  26.7305           26.3901            26.4546
R06      A_lag1        9.4134              12.2579       19.8097   8.9271   9.3817  11.2203   8.8980   9.8346            9.2592             9.7152
         B_rich        9.4134              12.2579           NaN   8.9271   9.3817  11.2203   8.8980   9.8346            9.2592             9.7152

## Tabel 3 - Rossmann: ablasi kebocoran `Customers`

Skala log dan skala asli dipisahkan ke dua blok kolom dengan penanda eksplisit.

In [5]:
if rossman is not None:
    t3 = (rossman.set_index(["feature_set", "model"])
          [["n_features", "test_RMSE", "test_MAE", "test_R2",
            "orig_RMSE", "orig_MSE", "orig_MAE", "orig_RMSPE", "orig_R2"]]
          .rename(columns={"test_RMSE": "RMSE (log)", "test_MAE": "MAE (log)",
                           "test_R2": "R2 (log)", "orig_RMSE": "RMSE (asli)",
                           "orig_MSE": "MSE (asli)", "orig_MAE": "MAE (asli)",
                           "orig_RMSPE": "RMSPE (asli)", "orig_R2": "R2 (asli)"}))
    emit(t3.round(4), "tab3_rossmann_leakage_ablation",
         "Dampak kebocoran fitur Customers pada Rossmann Store Sales. "
         "V0 memakai Customers kontemporer yang tidak tersedia pada horizon peramalan.")
    display(t3)

ditulis: tab3_rossmann_leakage_ablation.csv / .md / .tex   -- Dampak kebocoran fitur Customers pada Rossmann Store Sales. V0 memakai Customers kontemporer yang tidak tersedia pada horizon peramalan.


n_features  RMSE (log)  MAE (log)  R2 (log)  RMSE (asli)    MSE (asli)   MAE (asli)  RMSPE (asli)  R2 (asli)
feature_set                  model                                                                                                                                                  
V0_customers_contemporaneous SeasonalNaive(store x dow x promo median)          16    0.164794   0.121103  0.842079  1274.631692  1.624686e+06   850.470064      0.157047   0.833536
                             LR                                                 16    0.215723   0.164838  0.729386  3143.046045  9.878738e+06  1295.489030      0.245372  -0.012165
                             XGBoost                                            16    0.086977   0.064749  0.956009   676.441795  4.575735e+05   459.210156      0.084541   0.953118
                             LR+XGB (average)                                   16    0.132527   0.102508  0.897868  1367.901444  1.871154e+06   779.152749      0.134189   0.808284
                             LR-XGB (residual)                                  16    0.087098   0.064648  0.955886   713.263799  5.087452e+05   462.767428      0.084627   0.947875
V1_customers_dropped         SeasonalNaive(store x dow x promo median)          15    0.164794   0.121103  0.842079  1274.631692  1.624686e+06   850.470064      0.157047   0.833536
                             LR                                                 15    0.362929   0.281054  0.234048  2846.796698  8.104251e+06  1979.623781      0.405838   0.169647
                             XGBoost                                            15    0.212096   0.152441  0.738409  1656.855741  2.745171e+06  1081.079985      0.211801   0.718733
                             LR+XGB (average)                                   15    0.252343   0.192367  0.629712  2102.849556  4.421976e+06  1394.496280      0.256347   0.546929
                             LR-XGB (residual)                                  15    0.212463   0.152503  0.737503  1661.714560  2.761295e+06  1081.777186      0.211496   0.717081
V2_customers_lagged          SeasonalNaive(store x dow x promo median)          17    0.164794   0.121103  0.842079  1274.631692  1.624686e+06   850.470064      0.157047   0.833536
                             LR                                                 17    0.275699   0.212355  0.557993  2519.163845  6.346186e+06  1542.601119      0.303546   0.349776
                             XGBoost                                            17    0.141765   0.107070  0.883131  1111.439116  1.235297e+06   761.099609      0.137465   0.873433
                             LR+XGB (average)                                   17    0.185013   0.142843  0.800950  1561.038229  2.436840e+06  1041.090787      0.185353   0.750324
                             LR-XGB (residual)                                  17    0.142708   0.107669  0.881572  1150.314852  1.323224e+06   773.387623      0.138830   0.864424
V3_sales_lagged              SeasonalNaive(store x dow x promo median)          17    0.164794   0.121103  0.842079  1274.631692  1.624686e+06   850.470064      0.157047   0.833536
                             LR                                                 17    0.244949   0.186298  0.651092  2049.059427  4.198645e+06  1325.656568      0.275335   0.569811
                             XGBoost                                            17    0.131400   0.097975  0.899597  1017.831046  1.035980e+06   695.058059      0.132630   0.893855
                             LR+XGB (average)                                   17    0.166946   0.126307  0.837927  1351.883152  1.827588e+06   907.251565      0.173997   0.812747
                             LR-XGB (residual)                                  17    0.132214   0.098551  0.898349  1058.824313  1.121109e+06   704.838371      0.134127   0.885132

## Tabel 4 - Ringkasan uji Diebold-Mariano

In [6]:
dm_frames = []
for stem, label in [("exp01_pharma_daily_dm_test", "PharmaSales harian"),
                    ("exp02_pharma_weekly_dm_test", "PharmaSales mingguan"),
                    ("exp03_rossmann_leakage_ablation_dm_test", "Rossmann")]:
    path = RESULTS / f"{stem}.csv"
    if path.exists():
        d = pd.read_csv(path); d.insert(0, "eksperimen", label); dm_frames.append(d)

if dm_frames:
    t4 = pd.concat(dm_frames, ignore_index=True)
    emit(t4.set_index(["eksperimen"]), "tab4_diebold_mariano",
         "Uji Diebold-Mariano (koreksi Harvey-Leybourne-Newbold) metode usulan "
         "terhadap pembanding terkuat pada kondisi identik.")
    display(t4)
    if "p_value" in t4.columns:
        sig = t4[t4["p_value"] < 0.05]
        print(f"\nSelisih signifikan pada alpha=0.05: {len(sig)} dari {len(t4)} perbandingan")

ditulis: tab4_diebold_mariano.csv / .md / .tex   -- Uji Diebold-Mariano (koreksi Harvey-Leybourne-Newbold) metode usulan terhadap pembanding terkuat pada kondisi identik.


,eksperimen,category,feature_set,pembanding,RMSE usulan,RMSE pembanding,DM,p_value,signifikan (a=0.05),varian,RMSE usulan (log),RMSE pembanding (log),usulan lebih baik
0,PharmaSales harian,M01AB,A_lag1,GRNN,3.0338,2.9140,3.480,6.000000e-04,True,NaN,NaN,NaN,NaN
1,PharmaSales harian,M01AB,A_lag1,SeasonalNaive(s=7),3.0338,4.0449,-5.834,0.000000e+00,True,NaN,NaN,NaN,NaN
2,PharmaSales harian,M01AB,B_rich,GRNN,3.0338,2.9140,3.480,6.000000e-04,True,NaN,NaN,NaN,NaN
3,PharmaSales harian,M01AB,B_rich,SeasonalNaive(s=7),3.0338,4.0449,-5.834,0.000000e+00,True,NaN,NaN,NaN,NaN
4,PharmaSales harian,M01AE,A_lag1,RBFNN,2.3027,2.3035,-0.054,9.568000e-01,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,Rossmann,NaN,NaN,XGBoost,NaN,NaN,2.760,5.786301e-03,NaN,V1_customers_dropped,0.21246,0.21210,False
68,Rossmann,NaN,NaN,SeasonalNaive,NaN,NaN,-44.573,0.000000e+00,NaN,V2_customers_lagged,0.14271,0.16479,True
69,Rossmann,NaN,NaN,XGBoost,NaN,NaN,6.172,6.752570e-10,NaN,V2_customers_lagged,0.14271,0.14177,False
70,Rossmann,NaN,NaN,SeasonalNaive,NaN,NaN,-59.504,0.000000e+00,NaN,V3_sales_lagged,0.13221,0.16479,True



Selisih signifikan pada alpha=0.05: 34 dari 72 perbandingan


## Tabel 5 - Kartu reproduktifitas

Tabel ini menjawab langsung butir "insufficient reproducibility": versi pustaka,
seed, rentang tanggal setiap blok, ukuran setiap blok, dan hyperparameter terpilih
per model - semuanya dibaca dari berkas hasil, bukan diketik ulang.

In [7]:
frames = [d for d in [daily, weekly, rossman] if d is not None]
if frames:
    repro = pd.concat(frames, ignore_index=True)[
        ["category", "feature_set", "model", "n_features", "n_lags", "lag_rule",
         "n_train", "n_val", "n_test", "train_start", "train_end",
         "val_start", "val_end", "test_start", "test_end", "scaler", "seed",
         "n_grid", "params"]]
    emit(repro.set_index(["category", "feature_set", "model"]),
         "tab5_reproducibility_card",
         "Kartu reproduktifitas: konfigurasi lengkap setiap sel pada Tabel 1-3.")
    display(repro.head(20))

meta = {}
for stem in ["exp01_pharma_daily", "exp02_pharma_weekly",
             "exp03_rossmann_leakage_ablation"]:
    path = RESULTS / f"{stem}.meta.json"
    if path.exists():
        meta[stem] = json.loads(path.read_text())
(OUT / "environment.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=2)[:1500])

ditulis: tab5_reproducibility_card.csv / .md / .tex   -- Kartu reproduktifitas: konfigurasi lengkap setiap sel pada Tabel 1-3.


,category,feature_set,model,n_features,n_lags,lag_rule,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end,scaler,seed,n_grid,params
0,M01AB,A_lag1,Naive,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,0,{}
1,M01AB,A_lag1,SeasonalNaive(s=7),1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,0,{}
2,M01AB,A_lag1,LR,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,1,{}
3,M01AB,A_lag1,GRNN,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,standard,42,16,"{""sigma"": 50.0}"
4,M01AB,A_lag1,P_NN,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,standard,42,16,"{""sigma"": 0.2}"
5,M01AB,A_lag1,RBFNN,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,standard,42,60,"{""alpha"": 1.0, ""gamma"": 0.01, ""n_centers"": 5}"
6,M01AB,A_lag1,XGBoost,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,12,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0..."
7,M01AB,A_lag1,LR+XGB (average),1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,12,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.1..."
8,M01AB,A_lag1,LR-XGB (residual),1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,12,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0..."
9,M01AB,B_rich,Naive,1,1,pacf_train,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,none,42,0,{}


{
  "exp01_pharma_daily": {
    "experiment": "exp01_pharma_daily",
    "n_rows": 152,
    "protocol": {
      "split_ratios": [
        0.7,
        0.15,
        0.15
      ],
      "tuning": "validation-only, refit on train+val",
      "feature_sets": [
        "A_lag1",
        "B_rich"
      ]
    },
    "environment": {
      "python": "3.10.11",
      "platform": "Windows-10-10.0.22621-SP0",
      "numpy": "2.2.6",
      "pandas": "2.3.3",
      "seed": "42",
      "sklearn": "1.7.2",
      "xgboost": "3.2.0",
      "statsmodels": "0.14.6"
    }
  },
  "exp02_pharma_weekly": {
    "experiment": "exp02_pharma_weekly",
    "n_rows": 152,
    "protocol": {
      "split_ratios": [
        0.7,
        0.15,
        0.15
      ],
      "tuning": "validation-only, refit on train+val",
      "feature_sets": [
        "A_lag1",
        "B_rich"
      ]
    },
    "environment": {
      "python": "3.10.11",
      "platform": "Windows-10-10.0.22621-SP0",
      "numpy": "2.2.6",
      "pan

## Pemeriksaan konsistensi terakhir

Sebelum angka dipindahkan ke naskah, sel ini memverifikasi ulang aritmetikanya.
Reviewer menemukan RMSE 525,994 dilaporkan bersama MSE 276.669,25 (tidak mengkuadrat
secara eksak). Pemeriksaan ini membuat kesalahan sejenis mustahil lolos.

In [8]:
problems = []
for name, df in [("daily", daily), ("weekly", weekly), ("rossmann", rossman)]:
    if df is None:
        continue
    for prefix in ["val_", "test_", "orig_"]:
        rmse_col, mse_col = f"{prefix}RMSE", f"{prefix}MSE"
        if rmse_col not in df.columns or mse_col not in df.columns:
            continue
        bad = df[~np.isclose(df[rmse_col] ** 2, df[mse_col], rtol=1e-9, atol=1e-12)]
        if len(bad):
            problems.append((name, prefix, len(bad)))
    # tidak boleh ada sel kosong pada kolom metrik utama
    empties = df[["test_RMSE", "test_MSE", "test_MAE"]].isna().sum().sum()
    if empties:
        problems.append((name, "sel metrik kosong", int(empties)))

if problems:
    print("MASALAH DITEMUKAN:")
    for p_ in problems:
        print("  ", p_)
else:
    print("Lolos: RMSE = sqrt(MSE) secara eksak di semua baris; "
          "tidak ada sel metrik yang kosong.")

Lolos: RMSE = sqrt(MSE) secara eksak di semua baris; tidak ada sel metrik yang kosong.
